<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/06_Interview_Prep/ai_engineer/02_rag_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG From Scratch — Live Interview Challenge

Build a complete RAG pipeline in 45 minutes. Every component explained from first principles, with the exact code interviewers expect you to write.

**The Challenge:** *Build a document Q&A system. Given a set of documents, answer questions based only on their content.*

**Components:** Document ingestion → Chunking → Embedding → Indexing → Retrieval → Generation → Evaluation

## 1. Document Ingestion and Chunking

In [ ]:
from __future__ import annotations
import re
import hashlib
from dataclasses import dataclass, field
from typing import Iterator

@dataclass
class Document:
    content: str
    metadata: dict = field(default_factory=dict)

    @property
    def doc_id(self) -> str:
        return hashlib.md5(self.content.encode()).hexdigest()[:8]

@dataclass
class Chunk:
    text: str
    doc_id: str
    chunk_index: int
    metadata: dict = field(default_factory=dict)

    @property
    def chunk_id(self) -> str:
        return f'{self.doc_id}_{self.chunk_index}'

def chunk_by_words(
    doc: Document,
    chunk_size: int = 200,    # words per chunk
    overlap: int = 50,        # overlapping words between chunks
) -> list[Chunk]:
    """Split document into overlapping word-count chunks."""
    words = doc.content.split()
    chunks = []
    step = chunk_size - overlap

    for i, start in enumerate(range(0, len(words), step)):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunk_text = ' '.join(chunk_words)
        chunks.append(Chunk(
            text=chunk_text,
            doc_id=doc.doc_id,
            chunk_index=i,
            metadata={**doc.metadata, 'word_start': start, 'word_end': start + len(chunk_words)}
        ))

    return chunks

def chunk_by_sentences(
    doc: Document,
    max_sentences: int = 5,
    overlap_sentences: int = 1,
) -> list[Chunk]:
    """Split at sentence boundaries (more semantically coherent than word splitting)."""
    # Simple sentence splitter (use spacy/nltk in production)
    sentences = re.split(r'(?<=[.!?])\s+', doc.content.strip())
    chunks = []
    step = max_sentences - overlap_sentences

    for i, start in enumerate(range(0, len(sentences), step)):
        chunk_sentences = sentences[start:start + max_sentences]
        if not chunk_sentences:
            break
        chunks.append(Chunk(
            text=' '.join(chunk_sentences),
            doc_id=doc.doc_id,
            chunk_index=i,
            metadata={**doc.metadata, 'sent_start': start}
        ))

    return chunks

# Sample documents (in real interview: load from PDFs/URLs)
sample_docs = [
    Document(
        content="""Python is a high-level, general-purpose programming language. Its design philosophy
        emphasizes code readability through the use of significant indentation. Python is dynamically
        typed and garbage-collected. It supports multiple programming paradigms, including structured,
        object-oriented and functional programming. It was created by Guido van Rossum and first
        released in 1991. Python consistently ranks as one of the most popular programming languages.""",
        metadata={'source': 'python_wiki', 'topic': 'Python'}
    ),
    Document(
        content="""Machine learning is a subset of artificial intelligence that enables systems to learn
        and improve from experience without being explicitly programmed. It focuses on developing
        computer programs that can access data and use it to learn for themselves. The primary aim
        of ML is to allow computers to learn automatically without human intervention. Key types
        include supervised learning, unsupervised learning, and reinforcement learning.""",
        metadata={'source': 'ml_wiki', 'topic': 'ML'}
    ),
    Document(
        content="""Large language models (LLMs) are machine learning models that can perform many kinds
        of natural language processing tasks. LLMs are artificial neural networks that use transformer
        architectures. They are pre-trained on vast amounts of text data using self-supervised learning.
        GPT-4 from OpenAI and Claude from Anthropic are examples of frontier LLMs. These models
        require enormous computational resources to train, often consuming thousands of GPU-hours.""",
        metadata={'source': 'llm_wiki', 'topic': 'LLMs'}
    ),
]

all_chunks = []
for doc in sample_docs:
    chunks = chunk_by_sentences(doc, max_sentences=3, overlap_sentences=1)
    all_chunks.extend(chunks)

print(f'Ingested {len(sample_docs)} documents → {len(all_chunks)} chunks')
print()
for chunk in all_chunks[:3]:
    print(f'Chunk {chunk.chunk_id}: "{chunk.text[:80]}..."')
    print(f'  Metadata: {chunk.metadata}')
    print()

## 2. Embedding and Vector Index

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dataclasses import dataclass
from typing import Protocol

# In production: use OpenAI/Cohere/HuggingFace embeddings
# Here we use TF-IDF as a lightweight substitute (no API key needed)
# The interface is identical — swap embed() for real embeddings

class EmbeddingModel(Protocol):
    def embed(self, texts: list[str]) -> np.ndarray: ...

class TFIDFEmbedder:
    """TF-IDF sparse → dense embeddings (interview-safe, no API key)."""
    def __init__(self, max_features: int = 1000):
        self.vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
        self._fitted = False

    def fit(self, texts: list[str]) -> 'TFIDFEmbedder':
        self.vectorizer.fit(texts)
        self._fitted = True
        return self

    def embed(self, texts: list[str]) -> np.ndarray:
        return self.vectorizer.transform(texts).toarray()

@dataclass
class IndexedChunk:
    chunk: 'Chunk'
    embedding: np.ndarray

class VectorIndex:
    """Simple in-memory vector index with cosine similarity search."""

    def __init__(self, embedder: TFIDFEmbedder):
        self.embedder = embedder
        self._index: list[IndexedChunk] = []

    def add_chunks(self, chunks: list['Chunk']) -> None:
        """Embed and store chunks."""
        texts = [c.text for c in chunks]
        embeddings = self.embedder.embed(texts)
        for chunk, emb in zip(chunks, embeddings):
            self._index.append(IndexedChunk(chunk=chunk, embedding=emb))

    def search(self, query: str, k: int = 3) -> list[tuple[float, 'Chunk']]:
        """Return top-k most similar chunks with their scores."""
        query_emb = self.embedder.embed([query])
        all_embeddings = np.vstack([item.embedding for item in self._index])
        scores = cosine_similarity(query_emb, all_embeddings)[0]
        top_k_idx = np.argsort(scores)[::-1][:k]
        return [(scores[i], self._index[i].chunk) for i in top_k_idx]

    def __len__(self) -> int:
        return len(self._index)

# Build the index
all_texts = [c.text for c in all_chunks]
embedder = TFIDFEmbedder(max_features=500)
embedder.fit(all_texts)

index = VectorIndex(embedder)
index.add_chunks(all_chunks)

print(f'Vector index built: {len(index)} chunks indexed')
print()

# Test retrieval
test_queries = [
    'Who created Python?',
    'What is machine learning?',
    'examples of large language models',
]

for query in test_queries:
    results = index.search(query, k=2)
    print(f'Query: "{query}"')
    for score, chunk in results:
        print(f'  [{score:.3f}] ({chunk.metadata["topic"]}) "{chunk.text[:80]}..."')
    print()

print('NOTE: In production, replace TFIDFEmbedder with:')
print('  from openai import OpenAI')
print('  client.embeddings.create(model="text-embedding-3-small", input=texts)')
print('  → Returns 1536-dim vectors capturing semantic meaning, not just keywords')

## 3. RAG Pipeline — Full Integration

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional

@dataclass
class RAGResponse:
    question: str
    answer: str
    retrieved_chunks: list[tuple[float, 'Chunk']]
    context_used: str

class RAGPipeline:
    """Complete RAG pipeline: retrieve + generate."""

    def __init__(
        self,
        index: VectorIndex,
        k: int = 3,
        min_score: float = 0.05,  # below this = no relevant context
    ):
        self.index = index
        self.k = k
        self.min_score = min_score

    def _build_context(self, chunks: list[tuple[float, 'Chunk']]) -> str:
        """Format retrieved chunks into context string."""
        parts = []
        for i, (score, chunk) in enumerate(chunks):
            source = chunk.metadata.get('source', 'unknown')
            parts.append(f'[Source {i+1}: {source} (relevance: {score:.2f})]\n{chunk.text}')
        return '\n\n'.join(parts)

    def _build_prompt(self, question: str, context: str) -> str:
        return f"""You are a helpful assistant. Answer the question based ONLY on the provided context.
If the answer is not in the context, say "I don't have enough information to answer this."
Do not make up information.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    def answer(self, question: str, llm_fn=None) -> RAGResponse:
        """Retrieve context and generate answer."""
        # Step 1: Retrieve
        retrieved = self.index.search(question, k=self.k)
        relevant = [(score, chunk) for score, chunk in retrieved if score >= self.min_score]

        if not relevant:
            return RAGResponse(
                question=question,
                answer="I don't have enough information to answer this.",
                retrieved_chunks=[],
                context_used=''
            )

        # Step 2: Build context
        context = self._build_context(relevant)
        prompt = self._build_prompt(question, context)

        # Step 3: Generate (use LLM in production, mock here)
        if llm_fn:
            answer = llm_fn(prompt)
        else:
            # Mock: extract first sentence of most relevant chunk
            best_chunk_text = relevant[0][1].text
            answer = f'[MOCK] Based on the retrieved context: {best_chunk_text[:200]}...'

        return RAGResponse(
            question=question,
            answer=answer,
            retrieved_chunks=relevant,
            context_used=context
        )

# Use the pipeline
rag = RAGPipeline(index, k=3, min_score=0.05)

questions = [
    'Who created Python and when was it first released?',
    'What are the types of machine learning?',
    'What companies make frontier LLMs?',
    'What is the population of Mars?',  # Out-of-scope question
]

print('RAG Pipeline Responses:')
print('='*60)
for q in questions:
    response = rag.answer(q)
    print(f'Q: {q}')
    print(f'A: {response.answer[:200]}')
    print(f'Retrieved: {len(response.retrieved_chunks)} chunks, '
          f'top score={response.retrieved_chunks[0][0]:.3f if response.retrieved_chunks else 0}')
    print()

# Production upgrade path
print('To use a real LLM (swap 3 lines):')
print('''
from openai import OpenAI
client = OpenAI()

def llm_fn(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content

rag.answer(question, llm_fn=llm_fn)
''')

## 4. RAG Evaluation — Measuring Quality Without Ground Truth

In [ ]:
import numpy as np

# Automated RAG evaluation without calling LLM APIs
# Production: use RAGAS library or LLM-as-judge

def evaluate_retrieval(
    question: str,
    retrieved_chunks: list,
    ground_truth_text: str,
) -> dict:
    """Evaluate retrieval quality using keyword overlap (proxy for real evaluation)."""
    if not retrieved_chunks:
        return {'recall': 0.0, 'precision': 0.0, 'mrr': 0.0}

    # Token overlap between ground truth and retrieved chunks
    gt_words = set(ground_truth_text.lower().split())

    # Recall: is GT content covered by any retrieved chunk?
    all_retrieved_text = ' '.join(c.text for _, c in retrieved_chunks).lower()
    retrieved_words = set(all_retrieved_text.split())
    recall = len(gt_words & retrieved_words) / len(gt_words) if gt_words else 0.0

    # Precision: what fraction of retrieved content is relevant?
    precision_scores = []
    for _, chunk in retrieved_chunks:
        chunk_words = set(chunk.text.lower().split())
        # Avoid empty sets
        if chunk_words and gt_words:
            overlap = len(chunk_words & gt_words) / len(chunk_words)
            precision_scores.append(overlap)
    precision = np.mean(precision_scores) if precision_scores else 0.0

    # MRR: position of first relevant chunk
    mrr = 0.0
    for rank, (score, chunk) in enumerate(retrieved_chunks, 1):
        chunk_words = set(chunk.text.lower().split())
        if len(chunk_words & gt_words) / max(len(chunk_words), 1) > 0.1:
            mrr = 1.0 / rank
            break

    return {'recall': recall, 'precision': precision, 'mrr': mrr}

def faithfulness_score(answer: str, context: str) -> float:
    """Proxy for faithfulness: are answer words present in context?"""
    # Remove stopwords (simplified)
    stopwords = {'the', 'a', 'an', 'is', 'was', 'are', 'were', 'be', 'been',
                  'have', 'has', 'had', 'do', 'does', 'did', 'to', 'of', 'in',
                  'for', 'on', 'with', 'by', 'from', 'at', 'this', 'that'}
    answer_words = set(answer.lower().split()) - stopwords
    context_words = set(context.lower().split()) - stopwords
    if not answer_words:
        return 1.0
    return len(answer_words & context_words) / len(answer_words)

# Golden test set
golden_dataset = [
    {
        'question': 'Who created Python?',
        'ground_truth': 'Python was created by Guido van Rossum.',
    },
    {
        'question': 'What are the types of machine learning?',
        'ground_truth': 'Supervised learning, unsupervised learning, and reinforcement learning.',
    },
    {
        'question': 'What companies make LLMs?',
        'ground_truth': 'OpenAI makes GPT-4 and Anthropic makes Claude.',
    },
]

print('RAG Evaluation on Golden Dataset:')
print(f'{"Question":<40} {"Recall":>8} {"Precision":>10} {"MRR":>6}')
print('-' * 70)

all_scores = []
for item in golden_dataset:
    response = rag.answer(item['question'])
    metrics = evaluate_retrieval(item['question'], response.retrieved_chunks, item['ground_truth'])
    all_scores.append(metrics)
    print(f'{item["question"][:38]:<40} {metrics["recall"]:>8.3f} {metrics["precision"]:>10.3f} {metrics["mrr"]:>6.3f}')

print('-' * 70)
avg = {k: np.mean([s[k] for s in all_scores]) for k in all_scores[0]}
print(f'{"AVERAGE":<40} {avg["recall"]:>8.3f} {avg["precision"]:>10.3f} {avg["mrr"]:>6.3f}')

print()
print('Production: Use RAGAS for proper LLM-based evaluation:')
print('  from ragas import evaluate')
print('  from ragas.metrics import faithfulness, answer_relevancy, context_recall')
print('  result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_recall])')

## 5. Advanced RAG Upgrades — What to Mention in Interviews

In [ ]:
import numpy as np

# ── Hybrid Search: BM25 + Semantic ──────────────────────────────────────

def bm25_score(
    query_terms: list[str],
    doc: str,
    corpus: list[str],
    k1: float = 1.5,
    b: float = 0.75,
) -> float:
    """BM25 relevance score. k1=1.5, b=0.75 are standard defaults."""
    doc_words = doc.lower().split()
    doc_len = len(doc_words)
    avg_doc_len = np.mean([len(d.split()) for d in corpus])
    n_docs = len(corpus)

    score = 0.0
    for term in query_terms:
        term_lower = term.lower()
        tf = doc_words.count(term_lower)
        df = sum(1 for d in corpus if term_lower in d.lower().split())
        if df == 0:
            continue
        idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1)
        tf_normalized = tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len / avg_doc_len))
        score += idf * tf_normalized

    return score

def hybrid_search(
    query: str,
    chunks: list['Chunk'],
    index: VectorIndex,
    alpha: float = 0.5,  # 0=pure BM25, 1=pure semantic
    k: int = 3,
) -> list[tuple[float, 'Chunk']]:
    """Hybrid retrieval: linear combination of BM25 and semantic scores."""
    corpus_texts = [c.text for c in chunks]
    query_terms = query.split()

    # BM25 scores
    bm25_scores = np.array([bm25_score(query_terms, c.text, corpus_texts) for c in chunks])

    # Semantic scores from index
    semantic_results = index.search(query, k=len(chunks))
    semantic_scores_dict = {chunk.chunk_id: score for score, chunk in semantic_results}
    semantic_scores = np.array([semantic_scores_dict.get(c.chunk_id, 0.0) for c in chunks])

    # Normalize both to [0, 1]
    def normalize(arr):
        min_v, max_v = arr.min(), arr.max()
        return (arr - min_v) / (max_v - min_v + 1e-10)

    combined = (1 - alpha) * normalize(bm25_scores) + alpha * normalize(semantic_scores)

    top_k_idx = np.argsort(combined)[::-1][:k]
    return [(combined[i], chunks[i]) for i in top_k_idx]

print('Hybrid Search (BM25 + Semantic, α=0.5):')
test_q = 'transformer architecture GPU training'
hybrid_results = hybrid_search(test_q, all_chunks, index, alpha=0.5)
for score, chunk in hybrid_results:
    print(f'  [{score:.3f}] ({chunk.metadata["topic"]}) "{chunk.text[:90]}..."')
print()

# ── Contextual Compression ──────────────────────────────────────────────

def extract_relevant_sentences(chunk_text: str, query: str, min_overlap: int = 2) -> str:
    """Keep only sentences from chunk that share words with query (proxy for LLM compression)."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', chunk_text)
    query_words = set(query.lower().split()) - {'what', 'is', 'are', 'the', 'a', 'an', 'how'}
    relevant = [s for s in sentences
                if len(set(s.lower().split()) & query_words) >= min_overlap]
    return ' '.join(relevant) if relevant else chunk_text

print('Contextual Compression (reduce noise in retrieved chunks):')
query_ex = 'Who created Python and when'
results = index.search(query_ex, k=1)
if results:
    _, best_chunk = results[0]
    compressed = extract_relevant_sentences(best_chunk.text, query_ex)
    print(f'  Original ({len(best_chunk.text.split())} words): "{best_chunk.text}"')
    print(f'  Compressed ({len(compressed.split())} words): "{compressed}"')
    print()

print('RAG Upgrade Checklist for interviews:')
upgrades = [
    ('Better chunking', 'Sentence-boundary aware, parent-child chunks'),
    ('Better embeddings', 'Swap TF-IDF for text-embedding-3-small or BGE-M3'),
    ('Hybrid search', 'BM25 + semantic with Reciprocal Rank Fusion'),
    ('Reranking', 'Cross-encoder (ms-marco-MiniLM) to re-score top-20 → top-5'),
    ('Query expansion', 'HyDE or multi-query to improve retrieval recall'),
    ('Contextual compression', 'LLM extracts only relevant parts of each chunk'),
    ('Metadata filtering', 'Pre-filter by date, author, document type before semantic search'),
    ('Evaluation', 'RAGAS metrics → target faithfulness > 0.85'),
]
for upgrade, description in upgrades:
    print(f'  {upgrade:<25}: {description}')

## Exercises

1. **Add metadata filtering:** Extend the `VectorIndex` to support pre-filtering chunks by metadata fields (e.g., `topic='LLMs'` before semantic search). Benchmark: compare retrieval quality with and without filtering on a test set of 20 questions tagged by topic.

2. **Implement HyDE:** Given a question, use an LLM (or a template) to generate a hypothetical answer document. Embed that hypothetical document and use it as the query vector instead of the original question. Test on 5 questions where direct query embedding retrieves irrelevant results.

3. **Build a multi-document summarizer:** Given 3-5 documents on the same topic, chunk and index them, then use iterative RAG to answer: (a) "Summarize all documents in 3 bullet points", (b) "What do these documents disagree on?". Handle context window limits gracefully.